# Marketing Campaign Conversion Analysis

## Project Overview

This project analyzes marketing campaign performance and customer behavior to identify the key factors driving customer conversion. The analysis combines exploratory data analysis (EDA) and machine learning techniques to uncover actionable marketing insights.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/digital_marketing_campaign_dataset.csv")



## Data Understanding

Before conducting the analysis, the dataset is explored to understand its structure, identify missing values, and examine the distribution of key variables.

In [ ]:
df.head()
df.describe()
df.isnull().sum()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df.head()

In [ ]:
df.columns


## Business question 1: Which channels and campaign types generate the highest conversion?

In [ ]:

#Conversion by campaign channel
df.groupby("CampaignChannel")["Conversion"].mean().sort_values(ascending=False)
#Referral channel generate highest conversion rate

In [ ]:
#Conversion by campaign type
df.groupby("CampaignType")["Conversion"].mean().sort_values(ascending=False)
#Conversion campaign type generate highest conversion rate

In [ ]:
#Visualization
sns.barplot(data=df, x="CampaignChannel", y="Conversion")
plt.xticks(rotation=45)
plt.show()

## Business question 2:  Which customer groups are most likely to convert?

In [ ]:


df.groupby("Gender")["Conversion"].mean()
#Male customers have higher conversion

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[18,25,35,45,55,100],
    labels=["18-25","26-35","36-45","46-55","55+"]
)

df.groupby("AgeGroup")["Conversion"].mean()
#Customers aged 36-45 have highest conversion rate

sns.boxplot(x="Conversion", y="Income", data=df)
plt.show()
#Customers who converted vs those who not converted have same income distribution


## Business Question 3: What behaviors indicated higher conversion?

In [ ]:


#Time on site vs conversion
sns.boxplot(x="Conversion", y="TimeOnSite", data=df)
plt.show()
#Customers who converted have higher time on site

#Email clicks vs conversion
sns.boxplot(x="Conversion", y="EmailClicks", data=df)
plt.show()
#Customers who converted have higher email clicks

## Business question 4: Does higher advertising spend improve conversion?

In [ ]:

sns.scatterplot(x="AdSpend", y="ConversionRate", data=df)
plt.show()
#Ad spend showed little clear relationship with conversion rate, indicating that increased budget alone does not guarantee stronger campaign performance.

## Business question 5: Which campaign combinations perform best?

In [ ]:

pivot = df.pivot_table(
    values="Conversion",
    index="CampaignChannel",
    columns="CampaignType",
    aggfunc="mean"
)

sns.heatmap(pivot, annot=True, cmap="coolwarm")
plt.show()

#PPC channel - Conversion type and SEO-channel - Conversion type perform best

## Business question 6: Are loyal customers more likely to convert?

In [ ]:

sns.boxplot(x="Conversion", y="LoyaltyPoints", data=df)
plt.show()

#Customers who converted have higher loyalty points

## Business question 7: Do email opens, email clicks drive conversions? 

In [ ]:

df.groupby("Conversion")["EmailOpens"].mean()
#Customers who converted have higher average email opens

df.groupby("Conversion")["EmailClicks"].mean()
#Customers who converted have higher average email clicks


## Business question 8: What factors had strong/weak correlation with conversion?

In [ ]:

corr = df.corr(numeric_only=True)

plt.figure(figsize=(12,8))
sns.heatmap(corr, cmap="coolwarm")
plt.show()

##Email clicks and opens show positive correlation with conversion
#Previous purchases and loyalty points also show positive correlation with conversion
#PagesPerVisit,TimeOnSite, WebsiteVisits also show positive correlation with conversion
#Age, income show weak correlation with conversion
#Behavioral engagement metrics such as email interactions, website activity, and loyalty indicators demonstrated stronger relationships with conversion than demographic variables such as age and income.

## Business question 9: Predict whether a customer converts

In [ ]:

# Predictive Analytics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df.drop(["CustomerID", "Conversion"], axis=1)

y = df["Conversion"]

label_encoder = LabelEncoder()

categorical_cols = X.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    X[col] = label_encoder.fit_transform(X[col])

    X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
    log_model = LogisticRegression(max_iter=1000,    class_weight="balanced")

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_log))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

#Class 0: recall 0.01: the model only identified 1% of actual non-converters correctly.

#Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

rf_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

#logistic regression recall for class 0 is 0.56; random forest recall for class 0 is 0.05

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(10)

#Which variables were most important for predicting customer conversion: customer engagement behaviors


plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top Features Driving Conversion")

plt.show()

#TimeOnSite, PagesPerVisit: Customers who spend more time browsing and exploring the website are more likely to convert.
#Clickthroughrate: Customers who actively interact with advertisements are more likely to convert.
#LoyaltyPoints, PreviousPurchases: existing loyal customers are more likely to convert again.
#email opens, email clicks: email engagement contributes to customer conversion probability.

sample_customer = X_test.iloc[[0]]

prediction = log_model.predict(sample_customer)

print(prediction)

sample_customer

#the model predicted this customer will convert, because This customer has MANY characteristics associated with converters (high website visits
#long time on site, high email opens, high email clicks)

### Logistic Regression Results

The initial Logistic Regression model achieved high accuracy but struggled to identify non-converted customers due to class imbalance. A balanced version of the model improved minority-class detection.

### Random Forest Results

Random Forest achieved higher overall accuracy but demonstrated weaker performance in identifying non-converted customers compared with Logistic Regression.

### Feature Importance Analysis

Feature importance analysis revealed that customer engagement metrics were the strongest predictors of conversion.

Top predictors:
- TimeOnSite
- PagesPerVisit
- AdSpend
- ClickThroughRate
- LoyaltyPoints

These findings suggest that customer behavior is more influential than demographic characteristics when predicting conversion.

### Business Recommendations

Based on the analysis, the following recommendations are proposed:

- Increase website engagement initiatives to improve conversion rates.
- Prioritize customers with strong engagement signals.
- Strengthen loyalty programs to encourage repeat conversions.
- Improve email marketing effectiveness through higher engagement strategies.
- Focus on behavioral targeting rather than demographic segmentation alone.

### Conclusion

This analysis found that customer engagement and loyalty indicators were the strongest drivers of conversion. Machine learning models further confirmed that behavioral metrics such as TimeOnSite and PagesPerVisit were more predictive of conversion than demographic variables.